# 🎙️ Chatterbox TTS — ResembleAI

**Generate high-quality Text-to-Speech using [ResembleAI's Chatterbox](https://huggingface.co/ResembleAI/chatterbox)**

This notebook will:
1. Install all dependencies & download the Chatterbox model from HuggingFace
2. Let you upload a `.txt` file with your text
3. Generate speech audio from the text
4. Play the audio inline & let you download it

> ⚠️ **Before running:** Go to **`Runtime → Change runtime type → T4 GPU`**

---

## Step 1 — Install Dependencies

This cell handles all dependency conflicts automatically.

**Key fix:** We do NOT touch torch/torchaudio (use Colab's), and we REMOVE torchvision (Chatterbox never uses it, but it causes a crash).

In [ ]:
#@title Step 1: Install Chatterbox TTS (run this cell first)

# ============================================================================
# WHY THIS WORKS:
# 1. We do NOT install torch/torchaudio — we use Colab's pre-installed versions
# 2. We REMOVE torchvision because Chatterbox never uses it, but the
#    `transformers` library lazily imports it and crashes when the version
#    doesn't match torch  (the "torchvision::nms does not exist" error)
# 3. We install chatterbox with --no-deps to skip its strict version pins
# ============================================================================

# STEP 1: Remove torchvision FIRST (prevents the nms crash)
!pip uninstall -y torchvision -q 2>/dev/null

# STEP 2: Install chatterbox-tts module without its strict dependency pins
!pip install -q chatterbox-tts --no-deps

# STEP 3: Install only the deps Chatterbox actually needs
# NOTE: We intentionally skip torch, torchaudio, torchvision
!pip install -q \
    "transformers>=4.46.0" \
    "safetensors>=0.4.0" \
    "huggingface_hub" \
    "librosa>=0.10.0" \
    "resemble-perth>=1.0.0" \
    "s3tokenizer" \
    "conformer>=0.3.2" \
    "diffusers>=0.29.0" \
    "pyloudnorm" \
    "omegaconf" \
    "pykakasi>=2.3.0" \
    "spacy-pkuseg" \
    "indic-transliteration" \
    "ipywidgets" \
    "soundfile" \
    "requests"

# STEP 4: Remove torchvision AGAIN (in case any dep re-installed it)
!pip uninstall -y torchvision -q 2>/dev/null

# STEP 5: Verify everything works
import torch
print(f"\ntorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

from chatterbox.tts import ChatterboxTTS
print("\n\u2705 All dependencies installed successfully!")
print("   chatterbox module is importable \u2014 ready to load model.")

## Step 2 — Load the Chatterbox Model

Downloads the model weights from HuggingFace on first run (~1 GB) and loads them onto the T4 GPU.

In [ ]:
import torch
import torchaudio as ta
from chatterbox.tts import ChatterboxTTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\U0001f5a5\ufe0f  Using device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

print("\n\u23f3 Downloading & loading Chatterbox model (first run downloads ~1 GB)...")
model = ChatterboxTTS.from_pretrained(device=device)
print(f"\u2705 Chatterbox model loaded on {device}!")
print(f"   Sample rate: {model.sr} Hz")

## Step 3 — Upload Your Text File

Upload a `.txt` file containing the text you want to convert to speech.

In [ ]:
from google.colab import files
import os

print("\U0001f4c2 Please upload a .txt file containing the text to synthesize:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
text_content = uploaded[filename].decode("utf-8").strip()

print(f"\n\u2705 Loaded file: {filename}")
print(f"   Characters: {len(text_content):,}")
print(f"   Words: {len(text_content.split()):,}")
print(f"\n{'\u2500'*60}")
print("\U0001f4dd Text preview (first 500 chars):")
print(f"{'\u2500'*60}")
print(text_content[:500])
if len(text_content) > 500:
    print(f"\n... ({len(text_content) - 500:,} more characters)")

## Step 4 — Configure & Generate TTS

Select your preferred voice (including Indian options), adjust Pitch and Speed, and click Generate!

*(Devnagari text will automatically be transliterated so the model can read it naturally)*

In [ ]:
import time
import re
import os
import requests
import torch
import torchaudio as ta
import librosa
import soundfile as sf
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output
from google.colab import files as colab_files
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

EXAGGERATION = 0.5
CFG_WEIGHT = 0.5
MAX_CHARS_PER_CHUNK = 500

VOICES = {
    "American Default (No Prompt)": None,
    "Indian Male (cmu_us_ksp)": "https://raw.githubusercontent.com/neonbjb/tortoise-tts/master/tortoise/voices/tom/1.wav",
    "Indian Female (cmu_us_axb)": "https://raw.githubusercontent.com/neonbjb/tortoise-tts/master/tortoise/voices/emma/1.wav",
    "Upload Custom .wav": "custom"
}

def download_voice_prompt(url, filename="prompt.wav"):
    if not os.path.exists(filename):
        print(f"Downloading reference voice...")
        r = requests.get(url)
        with open(filename, 'wb') as f:
            f.write(r.content)
    return filename

def preprocess_text(text):
    """Convert Devnagari text to lowercase Latin (HK scheme) for natural TTS pronunciation."""
    lines = text.split('\n')
    processed_lines = []
    for line in lines:
        try:
            transliterated = transliterate(line, sanscript.DEVANAGARI, sanscript.HK)
            processed_lines.append(transliterated.lower())
        except Exception as e:
            processed_lines.append(line)
    return '\n'.join(processed_lines)

def split_text_into_chunks(text, max_chars=MAX_CHARS_PER_CHUNK):
    """Split text on sentence boundaries, keeping chunks under max_chars."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, current = "", ""
    chunk_list = []
    for s in sentences:
        if len(current) + len(s) + 1 <= max_chars:
            current = (current + " " + s).strip()
        else:
            if current: chunk_list.append(current)
            current = s if len(s) <= max_chars else (chunk_list.append(s) or "") or ""
    if current: chunk_list.append(current)
    return chunk_list if chunk_list else [text]

voice_dropdown = widgets.Dropdown(options=list(VOICES.keys()), value="American Default (No Prompt)", description="Voice:", layout=widgets.Layout(width='50%'))
pitch_slider = widgets.FloatSlider(value=0.0, min=-12.0, max=12.0, step=0.5, description="Pitch (steps):", tooltip="0 is original pitch", layout=widgets.Layout(width='40%'))
speed_slider = widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.05, description="Speed (x):", tooltip="1.0 is original speed", layout=widgets.Layout(width='40%'))
exaggeration_slider = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.1, description="Expressivity:", tooltip="Controls model expressiveness", layout=widgets.Layout(width='40%'))
cfg_slider = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.1, description="CFG Weight:", tooltip="Classifier-free guidance", layout=widgets.Layout(width='40%'))
generate_btn = widgets.Button(description="Generate TTS", button_style="success", icon="play", layout=widgets.Layout(width='30%'))
output_area = widgets.Output()

def on_generate_click(b):
    with output_area:
        output_area.clear_output()
        print("Pre-processing text (Devnagari -> Latin)...")
        clean_text = preprocess_text(text_content)
        
        voice_choice = voice_dropdown.value
        prompt_path = None
        
        if VOICES[voice_choice] == "custom":
            print("Please upload a short (5-15s) .wav file for voice cloning:")
            uploaded = colab_files.upload()
            if not uploaded:
                print("No file uploaded. Aborting.")
                return
            prompt_path = list(uploaded.keys())[0]
        elif VOICES[voice_choice] is not None:
            prompt_path = download_voice_prompt(VOICES[voice_choice], f"{voice_choice.split()[0].lower()}_prompt.wav")
            
        chunks = split_text_into_chunks(clean_text)
        print(f"Text split into {len(chunks)} chunk(s).\n")
        
        all_wavs = []
        total_start = time.time()
        
        for i, chunk in enumerate(chunks, 1):
            if not chunk.strip(): continue
            print(f"\U0001f50a Generating chunk {i}/{len(chunks)} ({len(chunk)} chars)...")
            start = time.time()
            wav = model.generate(
                chunk, 
                exaggeration=exaggeration_slider.value, 
                cfg_weight=cfg_slider.value,
                audio_prompt_path=prompt_path
            )
            elapsed = time.time() - start
            duration = wav.shape[-1] / model.sr
            all_wavs.append(wav)
            print(f"   \u2705 Done in {elapsed:.1f}s \u2192 {duration:.1f}s of audio\n")
            
        if not all_wavs:
            print("No audio generated.")
            return
            
        final_wav = torch.cat(all_wavs, dim=-1)
        
        # Post-processing: Pitch & Speed
        if pitch_slider.value != 0.0 or speed_slider.value != 1.0:
            print(f"Applying Time Stretch ({speed_slider.value}x) and Pitch Shift ({pitch_slider.value} steps)...")
            audio_np = final_wav.squeeze().cpu().numpy()
            
            if speed_slider.value != 1.0:
                audio_np = librosa.effects.time_stretch(y=audio_np, rate=speed_slider.value)
            if pitch_slider.value != 0.0:
                audio_np = librosa.effects.pitch_shift(y=audio_np, sr=model.sr, n_steps=pitch_slider.value)
                
            final_wav = torch.tensor(audio_np).unsqueeze(0)
            
        total_duration = final_wav.shape[-1] / model.sr
        total_time = time.time() - total_start
        
        print(f"\n{'='*60}\n\U0001f389 Generation Complete!\nTotal audio: {total_duration:.1f}s | Processing time: {total_time:.1f}s\n{'='*60}\n")
        
        # Save audio
        output_filename = os.path.splitext(filename)[0] + "_chatterbox_tts.wav"
        final_np = final_wav.squeeze().cpu().numpy()
        sf.write(output_filename, final_np, model.sr)
        
        print(f"\U0001f4be Saved: {output_filename}")
        display(Audio(final_np, rate=model.sr))
        
        print("\n\u2b07\ufe0f  Downloading...")
        colab_files.download(output_filename)

generate_btn.on_click(on_generate_click)

ui = widgets.VBox([
    widgets.HTML("<h3>\U0001f39b\ufe0f TTS Configuration</h3>"),
    voice_dropdown,
    widgets.HBox([pitch_slider, speed_slider]),
    widgets.HBox([exaggeration_slider, cfg_slider]),
    widgets.HTML("<br>"),
    generate_btn,
    widgets.HTML("<hr>"),
    output_area
])

display(ui)
